# Unit 13 solutions: evaluating the whole system (exposure, feedback loops, filter bubbles)

These reference solutions answer the questions a hit@10 cannot. **Who** gets recommended, and which books never are? Which **authors** get the exposure? Does a reader see a **narrower** slice of genres than they actually read? And what happens when the recommender is **retrained on reads it caused**?

Three honest headlines frame the work. First, **accuracy and concentration travel together**: the two most accurate paths (the two-tower and the Unit-10 hybrid) are among the most concentrated, with an item exposure Gini above 0.9 over the full catalog, while item-item CF is the least concentrated path. Second, a short, seeded **feedback loop** makes item-item narrower round by round -- coverage falls, the Gini rises, genre entropy falls -- and its clean hit@10 eventually falls too, because the model starts learning the recommender instead of the readers. Third, every mitigation (a popularity penalty, an exploration slot, a calibrated genre re-rank) is a **trade-off** with numbers on both sides, not a fix.

Every number is a **descriptive measurement on our synthetic data**, graded on `val` over the scoreboard's 500-reader val cohort. The sealed held-out split is never touched. The two-tower is fit **once**, in the setup cell, and two short 3-round feedback loops run in the whole notebook (Exercise 4, and the Challenge in Exercise 8).

In [1]:
import json
import math
from collections import Counter, defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    HybridRetrievalPath,
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    PopularityRetrievalPath,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    catalog_coverage,
    head_ids_from_counts,
    head_share,
    rank,
)
from bookrec.fairness import (
    author_catalog_share,
    author_exposure_gini,
    exposure_counts,
    exposure_gini,
    genre_entropy,
    genre_spread_vs_history,
    gini,
    long_tail_share,
    popularity_penalized,
    simulate_feedback_loop,
    val_cohort,
)

SEED = 0        # the two-tower's training seed (Unit 8)
LOOP_SEED = 0   # the feedback-loop simulation's seed (one generator for every draw)
K = 10          # every readout is at the top-10 cutoff

# --- the real interaction log + content (the same substrate as Units 1-11) -----------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")        # {item_id: Book} (2000 books)
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")     # {item_id: "space joined tokens"}
glove = bookrec.load_glove_subset()                                # the committed GloVe subset

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])      # readers the personalized paths cannot serve

# The interaction log as plain row dicts -- the row-mapping log the paths duck-type.
rows = pd.read_csv(interactions_path).to_dict("records")

# Train positives are each reader's `seen` set (their history); val positives are the target. We
# grade on `val`; no cell here may read the sealed held-out split.
seen_by = defaultdict(set)
val_pos_by = defaultdict(set)
for row in rows:
    if row["label"] != 1:
        continue
    if row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
    elif row["split"] == "val":
        val_pos_by[row["reader_id"]].add(row["item_id"])

# The scoreboard's val cohort: warm readers with an unseen val positive (500 readers, ascending).
cohort = val_cohort(rows, catalog_ids, cold_readers)

# Popularity (Unit 2) also gives us each book's TRAIN-positive count -- warm books only.
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
train_counts = popularity.counts                              # {item_id: train-positive count}
head = set(head_ids_from_counts(train_counts, catalog_ids))   # the head: top 10% by train count
zero_train = set(catalog_ids) - set(train_counts)             # books with NO train positive

# The paths built in Units 3, 4, 7, 8 and 10. The two-tower is the ONE slow fit (~15-25 s).
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)
hybrid = HybridRetrievalPath(lexical, two_tower).fit(rows, catalog=catalog_ids)   # pool 50, weight 0.7


def top_lists(path):
    # {reader: that reader's ranked top-K item ids} over the val cohort (seen books excluded).
    return {
        r: [c.item_id for c in rank(path.retrieve(r, {"seen": seen_by[r]}, K), n=K, exclude=seen_by[r])]
        for r in cohort
    }


def hit_of(lists):
    # hit@K of {reader: top-K ids}: the fraction of readers with an unseen val positive in their list.
    return float(np.mean([1.0 if set(top) & (val_pos_by[r] - seen_by[r]) else 0.0 for r, top in lists.items()]))

## Exercise 1 - The exposure Gini by hand, and why the never-shown books must count

I compute the Gini of the toy first, with every catalog book counted (zeros included), and get exactly 16/30. The Gini over only the three books that were shown is 4/18. Then I build item-item's and the two-tower's top-10 lists, count item-item's exposure per catalog book by hand, and confirm my formula matches `exposure_gini` to floating-point precision. The shown-only Ginis look far milder, and they are taken over very different numbers of books.

In [2]:
def gini_by_hand(values):
    # The sorted-form Gini over a list of non-negative numbers (0.0 if they sum to 0).
    x = np.sort(np.array(values, dtype=float))
    n = len(x)
    if x.sum() == 0:
        return 0.0
    return float(((2 * np.arange(1, n + 1) - n - 1) * x).sum() / (n * x.sum()))


toy_lists = {"A": [1, 2], "B": [1, 3], "C": [1, 2]}
toy_catalog = [1, 2, 3, 4, 5]
toy_counts = {item: 0 for item in toy_catalog}
for top in toy_lists.values():
    for item in top:
        toy_counts[item] += 1
toy_full_gini = gini_by_hand(list(toy_counts.values()))
toy_shown_gini = gini_by_hand([c for c in toy_counts.values() if c > 0])
toy_matches_helper = bool(abs(toy_full_gini - exposure_gini(toy_lists, toy_catalog)) < 1e-12)

item_item_lists = top_lists(item_item)
two_tower_lists = top_lists(two_tower)

item_item_exposure = {item_id: 0 for item_id in catalog_ids}
for top in item_item_lists.values():
    for item_id in top:
        item_item_exposure[item_id] += 1
item_item_gini_hand = gini_by_hand(list(item_item_exposure.values()))
item_item_gini = exposure_gini(item_item_lists, catalog_ids)
hand_matches_helper = bool(abs(item_item_gini_hand - item_item_gini) < 1e-12)
two_tower_gini = exposure_gini(two_tower_lists, catalog_ids)

item_item_shown = [c for c in item_item_exposure.values() if c > 0]
two_tower_shown = [c for c in exposure_counts(two_tower_lists, catalog_ids).values() if c > 0]
item_item_shown_gini = gini(item_item_shown)
two_tower_shown_gini = gini(two_tower_shown)

full_catalog_explanation = (
    "A shown-only Gini throws away the never-recommended books, which are the most unequal part of "
    "the picture, so it flatters a narrow path: the two-tower drops from about 0.93 to about 0.60. "
    "It also measures each path over a DIFFERENT n (980 books for item-item, 354 for the two-tower), "
    "so the two numbers answer different questions and cannot be compared. Counting the full catalog "
    "fixes n at 2,000 for every path, so a book nobody is shown counts against the path that hid it."
)

print(f"toy: counts {toy_counts}  full Gini={toy_full_gini:.3f} (16/30={16 / 30:.3f})  "
      f"shown-only Gini={toy_shown_gini:.3f} (4/18={4 / 18:.3f})  matches helper: {toy_matches_helper}")
print(f"item-item full-catalog Gini: by hand {item_item_gini_hand:.6f}, helper {item_item_gini:.6f} "
      f"(match: {hand_matches_helper})")
print(f"two-tower full-catalog Gini: {two_tower_gini:.3f}")
print(f"shown-only: item-item {item_item_shown_gini:.3f} over {len(item_item_shown)} books | "
      f"two-tower {two_tower_shown_gini:.3f} over {len(two_tower_shown)} books")
print(full_catalog_explanation)

assert abs(toy_full_gini - 16 / 30) < 1e-12 and abs(toy_shown_gini - 4 / 18) < 1e-12, "the toy checks on paper"
assert toy_matches_helper and hand_matches_helper, "the by-hand formula reproduces exposure_gini"
assert two_tower_gini > item_item_gini, "over the same n, the two-tower is the more concentrated path"
assert len(two_tower_shown) < len(item_item_shown), "the two-tower shows far fewer distinct books"
assert two_tower_shown_gini < two_tower_gini and item_item_shown_gini < item_item_gini, \
    "dropping the zeros always makes the Gini look milder"


toy: counts {1: 3, 2: 2, 3: 1, 4: 0, 5: 0}  full Gini=0.533 (16/30=0.533)  shown-only Gini=0.222 (4/18=0.222)  matches helper: True
item-item full-catalog Gini: by hand 0.772890, helper 0.772890 (match: True)
two-tower full-catalog Gini: 0.929
shown-only: item-item 0.537 over 980 books | two-tower 0.598 over 354 books
A shown-only Gini throws away the never-recommended books, which are the most unequal part of the picture, so it flatters a narrow path: the two-tower drops from about 0.93 to about 0.60. It also measures each path over a DIFFERENT n (980 books for item-item, 354 for the two-tower), so the two numbers answer different questions and cannot be compared. Counting the full catalog fixes n at 2,000 for every path, so a book nobody is shown counts against the path that hid it.


## Exercise 2 - The whole-system scoreboard: accuracy next to exposure, books and authors

I build all six paths' cohort top-10 lists (reusing Exercise 1's two) and put accuracy next to five exposure readouts in one table. The two most accurate paths are among the most concentrated, and item-item is the least concentrated. Then I read the author Gini against its books-per-author baseline: every path sits above it, and popularity hands one author a share of the slots far beyond their share of the catalog.

In [3]:
paths = {
    "popularity": popularity,
    "lexical": lexical,
    "semantic": semantic,
    "item-item": item_item,
    "two-tower": two_tower,
    "hybrid": hybrid,
}
built = {"item-item": item_item_lists, "two-tower": two_tower_lists}   # Exercise 1's lists, reused
lists_by_path = {name: built[name] if name in built else top_lists(path) for name, path in paths.items()}

scoreboard = pd.DataFrame([
    {
        "path": name,
        "hit@10": hit_of(lists),
        "coverage": catalog_coverage(list(lists.values()), catalog_ids),
        "head_share": head_share(list(lists.values()), head),
        "long_tail_share": long_tail_share(lists, head),
        "item_gini": exposure_gini(lists, catalog_ids),
        "author_gini": author_exposure_gini(lists, catalog),
    }
    for name, lists in lists_by_path.items()
]).sort_values("hit@10", ascending=False).reset_index(drop=True)

item_gini_of = dict(zip(scoreboard["path"], scoreboard["item_gini"]))
accurate_are_concentrated = bool(item_gini_of["two-tower"] > 0.9 and item_gini_of["hybrid"] > 0.9)
item_item_least_concentrated = bool(item_gini_of["item-item"] == min(item_gini_of.values()))

author_share = author_catalog_share(catalog)
catalog_share_gini = gini(author_share.values())
all_above_baseline = bool((scoreboard["author_gini"] > catalog_share_gini).all())


def top_author(lists):
    # (author_id, share of all exposure slots) for the most-exposed author in `lists`.
    slots = Counter(catalog[item].fields["author_id"] for top in lists.values() for item in top)
    author, count = slots.most_common(1)[0]
    return author, count / sum(slots.values())


popularity_top_author = top_author(lists_by_path["popularity"])
item_item_top_author = top_author(lists_by_path["item-item"])

author_explanation = (
    "The item Gini asks whether exposure piles onto a few BOOKS; the author Gini asks whether it piles "
    "onto a few CREATORS -- popularity gives its top author about 19% of all slots with about 0.5% of "
    "the books. But authors own different numbers of books, so even exposure exactly proportional to "
    "books-per-author would give an author Gini of about 0.21; every path's author Gini has to be read "
    "against that baseline, not against zero."
)

print(scoreboard.round(3))
print(f"two most accurate paths have item Gini > 0.9: {accurate_are_concentrated} | "
      f"item-item least concentrated: {item_item_least_concentrated}")
print(f"books-per-author baseline Gini: {catalog_share_gini:.3f} | every path above it: {all_above_baseline} "
      f"({len(author_share)} authors)")
for name, (author, share) in (("popularity", popularity_top_author), ("item-item", item_item_top_author)):
    print(f"{name:<10} top author {author}: {share:.3f} of the slots vs {author_share[author]:.3f} of the catalog")
print(author_explanation)

assert len(scoreboard) == 6 and list(scoreboard["path"])[:2] == ["hybrid", "two-tower"], \
    "the hybrid and the two-tower top the accuracy column"
assert accurate_are_concentrated and item_item_least_concentrated
assert all_above_baseline, "every path concentrates author exposure beyond the books-per-author baseline"
assert popularity_top_author[1] > 10 * author_share[popularity_top_author[0]], \
    "popularity's top author gets far more exposure than their catalog share"


         path  hit@10  coverage  head_share  long_tail_share  item_gini  \
0      hybrid   0.362     0.192       0.862            0.138      0.922   
1   two-tower   0.340     0.177       0.900            0.100      0.929   
2   item-item   0.252     0.490       0.491            0.509      0.773   
3     lexical   0.158     0.278       0.268            0.732      0.888   
4  popularity   0.108     0.008       1.000            0.000      0.995   
5    semantic   0.102     0.354       0.109            0.891      0.857   

   author_gini  
0        0.677  
1        0.687  
2        0.474  
3        0.600  
4        0.970  
5        0.573  
two most accurate paths have item Gini > 0.9: True | item-item least concentrated: True
books-per-author baseline Gini: 0.211 | every path above it: True (300 authors)
popularity top author 215: 0.189 of the slots vs 0.005 of the catalog
item-item  top author 255: 0.017 of the slots vs 0.004 of the catalog
The item Gini asks whether exposure piles onto 

## Exercise 3 - A three-point popularity-penalty curve: what does fairer exposure cost?

The penalty term is exactly $\lambda$ for the most-read book (its count *is* the max count) and exactly 0 for a zero-train book ($\log 1 = 0$). I then wrap item-item at $\lambda \in \{0, 0.3, 1.0\}$ and record the curve. $\lambda = 0$ reproduces plain item-item reader for reader; the Gini falls strictly and hit@10 never rises.

In [4]:
LAMBDAS = [0.0, 0.3, 1.0]   # three points of the lesson's predeclared grid {0, 0.1, 0.3, 0.5, 1.0}
max_count = max(train_counts.values())


def penalty_term(item_id, lam):
    # What the re-rank subtracts from a candidate's min-max score.
    return lam * math.log1p(train_counts.get(item_id, 0)) / math.log1p(max_count)


most_read_book = max(sorted(train_counts), key=lambda item: train_counts[item])
most_read_penalty = penalty_term(most_read_book, 0.3)
zero_train_penalty = penalty_term(min(zero_train), 0.3)

penalized_lists = {lam: top_lists(popularity_penalized(item_item, lam, train_counts, pool=50)) for lam in LAMBDAS}
penalty_curve = pd.DataFrame([
    {
        "lambda": lam,
        "hit@10": hit_of(lists),
        "item_gini": exposure_gini(lists, catalog_ids),
        "coverage": catalog_coverage(list(lists.values()), catalog_ids),
        "head_share": head_share(list(lists.values()), head),
    }
    for lam, lists in penalized_lists.items()
])
lambda0_reproduces = bool(penalized_lists[0.0] == item_item_lists)

curve_gini = list(penalty_curve["item_gini"])
curve_hit = list(penalty_curve["hit@10"])
gini_strictly_falls = all(b < a for a, b in zip(curve_gini, curve_gini[1:]))
hit_never_rises = all(b <= a for a, b in zip(curve_hit, curve_hit[1:]))
hit_deltas = [h - curve_hit[0] for h in curve_hit[1:]]
gini_deltas = [g - curve_gini[0] for g in curve_gini[1:]]

tradeoff_explanation = (
    "Lambda 0.3 buys a modest Gini drop (about -0.026) and a large cut in the head's share of the slots "
    "for a small hit cost (about -0.028). Lambda 1.0 roughly doubles the Gini drop (about -0.060) but "
    "more than halves hit@10 (about -0.140), so readers pay most of the price. No point on the curve is "
    "free: it is a menu of trade-offs, a Pareto readout, not a winner. Which point a real service uses "
    "is a value judgment about readers, authors and the business, made by people who own those "
    "consequences, not something a metric can decide."
)

print(f"most-read book {most_read_book} (count {max_count}): penalty {most_read_penalty} | "
      f"zero-train book {min(zero_train)}: penalty {zero_train_penalty}")
print(penalty_curve.round(3))
print(f"lambda 0 reproduces item-item: {lambda0_reproduces} | Gini strictly falls: {gini_strictly_falls} | "
      f"hit never rises: {hit_never_rises}")
print(f"hit deltas vs lambda 0: {[round(d, 3) for d in hit_deltas]} | "
      f"Gini deltas: {[round(d, 3) for d in gini_deltas]}")
print(tradeoff_explanation)

assert most_read_penalty == 0.3 and zero_train_penalty == 0.0, "the penalty is exactly lambda at the top and 0 at zero"
assert lambda0_reproduces, "lambda = 0 keeps the inner order"
assert gini_strictly_falls and hit_never_rises, "fairer exposure costs accuracy along the curve"
assert all(d < 0 for d in hit_deltas) and all(d < 0 for d in gini_deltas)


most-read book 1710 (count 65): penalty 0.3 | zero-train book 0: penalty 0.0
   lambda  hit@10  item_gini  coverage  head_share
0     0.0   0.252      0.773     0.490       0.491
1     0.3   0.224      0.747     0.505       0.411
2     1.0   0.112      0.713     0.511       0.183
lambda 0 reproduces item-item: True | Gini strictly falls: True | hit never rises: True
hit deltas vs lambda 0: [-0.028, -0.14] | Gini deltas: [-0.026, -0.06]
Lambda 0.3 buys a modest Gini drop (about -0.026) and a large cut in the head's share of the slots for a small hit cost (about -0.028). Lambda 1.0 roughly doubles the Gini drop (about -0.060) but more than halves hit@10 (about -0.140), so readers pay most of the price. No point on the curve is free: it is a menu of trade-offs, a Pareto readout, not a winner. Which point a real service uses is a value judgment about readers, authors and the business, made by people who own those consequences, not something a metric can decide.


## Exercise 4 - A short feedback loop: retraining on your own recommendations

My prediction, written before running the loop: **coverage will fall** (the model is retrained on reads of the books it already shows, so those books gain evidence and crowd the lists), and **clean hit will fall** after a round or two, because the new rows teach CF what the recommender showed rather than what readers want. The run confirms both: coverage falls and the Gini rises every round; clean hit ticks up once, then falls below round 0. Round 0 reproduces Exercise 1 and 2 exactly, and my own audit of the simulated rows finds the guard intact.

In [5]:
loop_prediction = (
    "Coverage will fall: each refit sees extra reads of books the model already showed, so those books "
    "gain co-reads and crowd the lists. Clean hit will fall after a round or two, because the new rows "
    "describe what the recommender showed, not what the readers actually like."
)

loop = simulate_feedback_loop(ItemItemRetrievalPath, rows, cohort=cohort, catalog=catalog,
                              seed=LOOP_SEED, rounds=3)
loop_table = pd.DataFrame([
    {"round": m.round, "coverage": m.coverage, "head_share": m.head_share, "gini": m.gini,
     "genre_entropy": m.genre_entropy, "clean_hit": m.clean_hit, "accepted": m.accepted}
    for m in loop
])

item_item_coverage = catalog_coverage(list(item_item_lists.values()), catalog_ids)
round0_matches = bool(
    abs(loop[0].gini - item_item_gini) < 1e-9
    and abs(loop[0].coverage - item_item_coverage) < 1e-9
    and abs(loop[0].clean_hit - hit_of(item_item_lists)) < 1e-9
)
coverage_fell = loop[0].coverage - loop[-1].coverage
gini_rose = loop[-1].gini - loop[0].gini
clean_hit_change = loop[-1].clean_hit - loop[0].clean_hit

simulated_rows = [row for m in loop for row in m.new_rows]
all_train_rows = all(row["split"] == "train" and row["label"] == 1 for row in simulated_rows)
guard_holds = all(row["item_id"] not in val_pos_by[row["reader_id"]] for row in simulated_rows)
read_pairs = [(row["reader_id"], row["item_id"]) for row in simulated_rows]
no_double_reads = len(set(read_pairs)) == len(read_pairs)

loop_explanation = (
    "Each round the model is retrained on reads it caused, so the books it already favoured gain counts "
    "and co-occurrences and are shown even more: coverage falls and the Gini rises, a rich-get-richer "
    "loop. Clean hit eventually falls because the new rows teach CF what the recommender showed, not "
    "what readers want, so the model drifts toward predicting itself. Without the guard, simulated "
    "readers would 'read' their own val books, the refit would be trained on some of the answers it is "
    "graded on, and clean hit would rise -- a leak (the lesson measured 0.288 vs 0.266), not learning."
)

print(f"prediction: {loop_prediction}")
print(loop_table.round(3))
print(f"round 0 reproduces Exercises 1-2: {round0_matches}")
print(f"coverage fell by {coverage_fell:+.3f} | Gini rose by {gini_rose:+.3f} | clean hit change {clean_hit_change:+.3f}")
print(f"simulated rows: {len(simulated_rows)} | all train/label-1: {all_train_rows} | "
      f"guard holds: {guard_holds} | no double reads: {no_double_reads}")
print(loop_explanation)

assert len(loop) == 4 and round0_matches, "round 0 is plain item-item on the original log"
assert coverage_fell > 0 and gini_rose > 0, "the loop concentrates exposure"
assert clean_hit_change < 0, "by round 3 the refit path scores below round 0 on the original val positives"
assert all_train_rows and guard_holds and no_double_reads, "the leakage guard and the bookkeeping hold"
assert sum(m.accepted for m in loop) == len(simulated_rows)


prediction: Coverage will fall: each refit sees extra reads of books the model already showed, so those books gain co-reads and crowd the lists. Clean hit will fall after a round or two, because the new rows describe what the recommender showed, not what the readers actually like.
   round  coverage  head_share   gini  genre_entropy  clean_hit  accepted
0      0     0.490       0.491  0.773          1.640      0.252      1422
1      1     0.474       0.533  0.802          1.361      0.266      1425
2      2     0.430       0.557  0.828          1.216      0.256      1323
3      3     0.412       0.545  0.835          1.184      0.206      1310
round 0 reproduces Exercises 1-2: True
coverage fell by +0.078 | Gini rose by +0.062 | clean hit change -0.046
simulated rows: 5480 | all train/label-1: True | guard holds: True | no double reads: True
Each round the model is retrained on reads it caused, so the books it already favoured gain counts and co-occurrences and are shown even more: cov

## Exercise 5 - Filter bubbles: genre spread of the recommendations vs the reader's history

For each reader I compare the genre entropy of their top-10 with the genre entropy of their own train history. Item-item is roughly as broad as the readers' histories; the two-tower and lexical are far narrower, for most readers. The loop from Exercise 4 then narrows item-item round by round.

In [6]:
history = {r: seen_by[r] for r in cohort}

spread = {name: genre_spread_vs_history(lists_by_path[name], history, catalog)
          for name in ("item-item", "two-tower", "lexical")}
bubble_table = pd.DataFrame([
    {
        "path": name,
        "mean_rec_entropy": float(np.mean([rec for rec, _ in pairs.values()])),
        "mean_history_entropy": float(np.mean([hist for _, hist in pairs.values()])),
        "frac_narrower": float(np.mean([1.0 if rec < hist else 0.0 for rec, hist in pairs.values()])),
    }
    for name, pairs in spread.items()
])

loop_entropy = [m.genre_entropy for m in loop]
loop_narrowing = loop_entropy[0] - min(loop_entropy[1:])

bubble_explanation = (
    "The two-tower packs each list with near neighbours from one region of the reader's taste, and "
    "lexical with text matches on the same topic, so both show most readers a narrower genre mix than "
    "the readers actually read. The feedback loop adds narrowing over time on top of that static "
    "bubble: the reads item-item causes reinforce the genres it already showed, so its entropy falls "
    "by about 0.46 bits in three rounds. Genre entropy only sees genre LABELS -- not authors, styles, "
    "eras or viewpoints -- so it is one proxy for a bubble, not the bubble itself."
)

print(bubble_table.round(3))
print(f"loop genre entropy by round: {[round(e, 3) for e in loop_entropy]} | narrowing {loop_narrowing:+.3f} bits")
print(bubble_explanation)

row_of = {row["path"]: row for row in bubble_table.to_dict("records")}
assert bubble_table["mean_history_entropy"].nunique() == 1, "the history side is the same for every path"
assert row_of["two-tower"]["frac_narrower"] > 0.7 and row_of["lexical"]["frac_narrower"] > 0.7, \
    "most readers see a narrower slice from the two-tower and from lexical"
assert row_of["item-item"]["mean_rec_entropy"] > row_of["two-tower"]["mean_rec_entropy"] > row_of["lexical"]["mean_rec_entropy"]
assert loop_narrowing > 0.3, "the feedback loop narrows item-item's lists"


        path  mean_rec_entropy  mean_history_entropy  frac_narrower
0  item-item             1.640                 1.564          0.420
1  two-tower             0.681                 1.564          0.836
2    lexical             0.300                 1.564          0.890
loop genre entropy by round: [1.64, 1.361, 1.216, 1.184] | narrowing +0.455 bits
The two-tower packs each list with near neighbours from one region of the reader's taste, and lexical with text matches on the same topic, so both show most readers a narrower genre mix than the readers actually read. The feedback loop adds narrowing over time on top of that static bubble: the reads item-item causes reinforce the genres it already showed, so its entropy falls by about 0.46 bits in three rounds. Genre entropy only sees genre LABELS -- not authors, styles, eras or viewpoints -- so it is one proxy for a bubble, not the bubble itself.


## Exercise 6 - An offline-metric limit: the val split reflects the logging policy

I pool the cohort's unseen val positives and find that a large share are head books, for a head that is 10% of the catalog. Then I split each path's hit by target type: both item-item and the two-tower hit head targets many times more often than tail targets.

In [7]:
val_targets = [item for r in cohort for item in val_pos_by[r] - seen_by[r]]
head_share_of_targets = float(np.mean([item in head for item in val_targets]))


def head_tail_hits(lists):
    # (head-target hit, tail-target hit, readers with a head target, readers with a tail target).
    head_hits = []
    tail_hits = []
    for r, top in lists.items():
        relevant = val_pos_by[r] - seen_by[r]
        if relevant & head:
            head_hits.append(1.0 if set(top) & relevant & head else 0.0)
        if relevant - head:
            tail_hits.append(1.0 if set(top) & (relevant - head) else 0.0)
    return float(np.mean(head_hits)), float(np.mean(tail_hits)), len(head_hits), len(tail_hits)


item_item_head_tail = head_tail_hits(lists_by_path["item-item"])
two_tower_head_tail = head_tail_hits(lists_by_path["two-tower"])
head_dominates = all(ht[0] >= 5 * ht[1] for ht in (item_item_head_tail, two_tower_head_tail))

logging_explanation = (
    "About 44% of the val targets sit in a head that holds 10% of the catalog, because the log was "
    "produced while those books were the visible ones. A path that recommends the head therefore hits "
    "those targets often and the tail targets rarely, so offline hit@10 on this log rewards "
    "concentration. But this is a measured association: we have no log collected under a different "
    "policy, so we cannot say how much of the score is inflation from the logging policy versus a real "
    "preference for those books."
)

print(f"val targets: {len(val_targets)} | head share of targets: {head_share_of_targets:.3f} (head = 10% of the catalog)")
for name, ht in (("item-item", item_item_head_tail), ("two-tower", two_tower_head_tail)):
    print(f"{name:<10} head-target hit {ht[0]:.3f} ({ht[2]} readers) vs tail-target hit {ht[1]:.3f} "
          f"({ht[3]} readers) -> {ht[0] / ht[1]:.0f}x")
print(f"head hit >= 5x tail hit for both: {head_dominates}")
print(logging_explanation)

assert head_share_of_targets > 0.3, "the val targets lean heavily toward a 10% head"
assert head_dominates
assert item_item_head_tail[2:] == two_tower_head_tail[2:], "the reader split depends only on the targets"


val targets: 1082 | head share of targets: 0.441 (head = 10% of the catalog)
item-item  head-target hit 0.356 (323 readers) vs tail-target hit 0.032 (370 readers) -> 11x
two-tower  head-target hit 0.517 (323 readers) vs tail-target hit 0.014 (370 readers) -> 38x
head hit >= 5x tail hit for both: True
About 44% of the val targets sit in a head that holds 10% of the catalog, because the log was produced while those books were the visible ones. A path that recommends the head therefore hits those targets often and the tail targets rarely, so offline hit@10 on this log rewards concentration. But this is a measured association: we have no log collected under a different policy, so we cannot say how much of the score is inflation from the logging policy versus a real preference for those books.


## Exercise 7 - The cold-start thread closes: which paths surface a never-read book?

I count, for each path, the distinct zero-train books anywhere in the cohort's lists. The three warm-only collaborative paths surface none; the two content paths reach many; the hybrid reaches a few through its lexical half. The feature tower's 109 is cited from the lesson, not refit.

In [8]:
zero_train_surfaced = {name: len({item for top in lists.values() for item in top} & zero_train)
                       for name, lists in lists_by_path.items()}
warm_only_zero = all(zero_train_surfaced[name] == 0 for name in ("popularity", "item-item", "two-tower"))
content_paths_reach = zero_train_surfaced["lexical"] > 0 and zero_train_surfaced["semantic"] > 0

CITED_FEATURE_TOWER_COLD = 109   # the lesson: zero-train books the feature tower surfaces on this cohort (of 818)

cold_explanation = (
    "Popularity, item-item and the ID-only two-tower learn only from train interactions, so a book with "
    "none has no count, no co-reads and an embedding that only ever received negative gradients -- it "
    "can never outscore a warm book. The content paths score text, so they reach hundreds of cold books, "
    "and the hybrid's 17 come from its lexical half (its two-tower half has none to offer); the feature "
    f"tower is the only learned path that reaches them ({CITED_FEATURE_TOWER_COLD} of 818, cited). For a "
    "debut author, that means zero exposure from the collaborative paths until readers find the book "
    "some other way -- which the collaborative paths then need before they will ever show it."
)

print(f"zero-train books: {len(zero_train)}")
for name, n in zero_train_surfaced.items():
    print(f"{name:<10} surfaces {n} distinct zero-train books")
print(f"warm-only paths surface 0: {warm_only_zero} | content paths reach cold books: {content_paths_reach} | "
      f"feature tower (cited): {CITED_FEATURE_TOWER_COLD}")
print(cold_explanation)

assert len(zero_train) == 818
assert warm_only_zero and content_paths_reach
assert 0 < zero_train_surfaced["hybrid"] <= zero_train_surfaced["lexical"], "the hybrid's cold books come from lexical"


zero-train books: 818
popularity surfaces 0 distinct zero-train books
lexical    surfaces 128 distinct zero-train books
semantic   surfaces 263 distinct zero-train books
item-item  surfaces 0 distinct zero-train books
two-tower  surfaces 0 distinct zero-train books
hybrid     surfaces 17 distinct zero-train books
warm-only paths surface 0: True | content paths reach cold books: True | feature tower (cited): 109
Popularity, item-item and the ID-only two-tower learn only from train interactions, so a book with none has no count, no co-reads and an embedding that only ever received negative gradients -- it can never outscore a warm book. The content paths score text, so they reach hundreds of cold books, and the hybrid's 17 come from its lexical half (its two-tower half has none to offer); the feature tower is the only learned path that reaches them (109 of 818, cited). For a debut author, that means zero exposure from the collaborative paths until readers find the book some other way -- 

## Exercise 8 - Challenge: an exploration slot inside the feedback loop

I run the second (and last) loop of the notebook with the exploration slot and compare round 3 with Exercise 4's loop. The slot keeps more of the catalog in play, a lower Gini and a far broader genre mix. Round 0's clean hit is identical by construction, and at round 3 the mitigated loop's clean hit is slightly *higher*.

In [9]:
explore_loop = simulate_feedback_loop(ItemItemRetrievalPath, rows, cohort=cohort, catalog=catalog,
                                      seed=LOOP_SEED, rounds=3, mitigation="explore")

coverage_gain = explore_loop[-1].coverage - loop[-1].coverage
gini_drop = explore_loop[-1].gini - loop[-1].gini
entropy_gain = explore_loop[-1].genre_entropy - loop[-1].genre_entropy
cost_0 = explore_loop[0].clean_hit - loop[0].clean_hit
cost_R = explore_loop[-1].clean_hit - loop[-1].clean_hit
explore_helps = bool(coverage_gain >= 0.04 and gini_drop <= -0.01)

explore_explanation = (
    "The slot puts books in front of readers that the model would never show, and the reads it causes "
    "enter the log, so the next refit has evidence about books outside its favourite set. Nine of ten "
    "slots still follow the model, so concentration slows rather than stops (the Gini still rises from "
    "round 0). cost_R comes out positive here because a more varied training log hurts CF's clean hit "
    "less than a self-reinforced one -- the accuracy cost of exploration is not guaranteed to be "
    "negative, and on a longer or different run it could go either way."
)

print(pd.DataFrame([
    {"loop": name, "coverage": m.coverage, "gini": m.gini, "genre_entropy": m.genre_entropy, "clean_hit": m.clean_hit}
    for name, m in (("plain, round 3", loop[-1]), ("explore, round 3", explore_loop[-1]))
]).round(3))
print(f"coverage gain {coverage_gain:+.3f} | Gini change {gini_drop:+.3f} | entropy gain {entropy_gain:+.3f} bits")
print(f"cost_0 = {cost_0:+.3f} | cost_R = {cost_R:+.3f} | exploration helps: {explore_helps}")
print(explore_explanation)

assert cost_0 == 0.0, "round 0 refits the same original log and scores the same unmitigated path"
assert explore_helps, "one random slot in ten widens coverage and lowers the Gini"
assert entropy_gain > 0
assert explore_loop[-1].gini > explore_loop[0].gini, "exploration slows the concentration; it does not stop it"


               loop  coverage   gini  genre_entropy  clean_hit
0    plain, round 3     0.412  0.835          1.184      0.206
1  explore, round 3     0.455  0.812          1.528      0.216
coverage gain +0.043 | Gini change -0.023 | entropy gain +0.344 bits
cost_0 = +0.000 | cost_R = +0.010 | exploration helps: True
The slot puts books in front of readers that the model would never show, and the reads it causes enter the log, so the next refit has evidence about books outside its favourite set. Nine of ten slots still follow the model, so concentration slows rather than stops (the Gini still rises from round 0). cost_R comes out positive here because a more varied training log hurts CF's clean hit less than a self-reinforced one -- the accuracy cost of exploration is not guaranteed to be negative, and on a longer or different run it could go either way.


## Exercise 9 - Challenge: a calibrated genre re-rank against the filter bubble

I write the greedy calibrated re-rank over the two-tower's pool of 50 and evaluate $\beta = 0$ (which must reproduce the two-tower's lists exactly) and $\beta = 0.5$. At $\beta = 0.5$ the lists' genre mix moves much closer to each reader's history for a small hit cost, with a little more coverage and a slightly lower Gini.

In [10]:
def genre_distribution(items):
    # Given: {genre: share} over the genre labels of `items` (each book contributes each of its genres).
    counts = Counter(g for item in items for g in catalog[item].fields["genres"].split(";") if g)
    total = sum(counts.values())
    return {g: c / total for g, c in counts.items()} if total else {}


def kl_to_history(p, items, alpha=0.01):
    # Given: KL(p || q~) in bits, q~ = (1 - alpha) * q(items) + alpha * p.
    q = genre_distribution(items)
    return sum(pg * math.log(pg / ((1 - alpha) * q.get(g, 0.0) + alpha * pg), 2) for g, pg in p.items())


def calibrated_top(reader, beta, pool=50):
    # Greedy calibrated re-rank of the two-tower's pool for one reader (a list of K item ids).
    candidates = two_tower.retrieve(reader, {"seen": seen_by[reader]}, pool)
    raw = {c.item_id: c.score for c in candidates}
    low, high = min(raw.values()), max(raw.values())
    score = {i: (s - low) / (high - low) if high > low else 1.0 for i, s in raw.items()}
    p = genre_distribution(seen_by[reader])
    chosen = []
    remaining = sorted(score)
    while len(chosen) < K and remaining:
        # max() keeps the FIRST maximum, so scanning ascending ids breaks ties toward the smaller id.
        best = max(remaining, key=lambda i: (1 - beta) * score[i] - beta * kl_to_history(p, chosen + [i]))
        chosen.append(best)
        remaining = [i for i in remaining if i != best]
    return chosen


calibrated_lists = {beta: {r: calibrated_top(r, beta) for r in cohort} for beta in (0.0, 0.5)}
beta0_reproduces = bool(calibrated_lists[0.0] == lists_by_path["two-tower"])

calibration_table = pd.DataFrame([
    {
        "beta": beta,
        "hit@10": hit_of(lists),
        "mean_genre_entropy": float(np.mean([genre_entropy(top, catalog) for top in lists.values()])),
        "mean_kl_to_history": float(np.mean([kl_to_history(genre_distribution(seen_by[r]), top)
                                             for r, top in lists.items()])),
        "coverage": catalog_coverage(list(lists.values()), catalog_ids),
        "item_gini": exposure_gini(lists, catalog_ids),
    }
    for beta, lists in calibrated_lists.items()
])

calibration_explanation = (
    "Calibration targets the PER-READER bubble: it makes each list's genre mix resemble that reader's own "
    "history, cutting the mean KL by about two thirds for a small hit cost (about -0.008). The popularity "
    "penalty targets CATALOG-WIDE concentration -- how exposure is spread over books -- and on item-item "
    "it cost far more hit for its Gini drop. Calibration widens coverage a little as a side effect, but "
    "barely moves the item Gini; neither is 'the' fix, because they measure and trade different things."
)

print(f"beta 0 reproduces the two-tower: {beta0_reproduces}")
print(calibration_table.round(3))
print(calibration_explanation)

cal = calibration_table.to_dict("records")
assert beta0_reproduces, "with beta = 0 the greedy pick is just the two-tower's own order"
assert cal[1]["mean_kl_to_history"] < cal[0]["mean_kl_to_history"], "calibration moves lists toward the history"
assert cal[1]["mean_genre_entropy"] > cal[0]["mean_genre_entropy"]
assert cal[1]["hit@10"] <= cal[0]["hit@10"] + 1e-12


beta 0 reproduces the two-tower: True
   beta  hit@10  mean_genre_entropy  mean_kl_to_history  coverage  item_gini
0   0.0   0.340               0.681               1.688     0.177      0.929
1   0.5   0.332               1.219               0.548     0.230      0.913
Calibration targets the PER-READER bubble: it makes each list's genre mix resemble that reader's own history, cutting the mean KL by about two thirds for a small hit cost (about -0.008). The popularity penalty targets CATALOG-WIDE concentration -- how exposure is spread over books -- and on item-item it cost far more hit for its Gini drop. Calibration widens coverage a little as a side effect, but barely moves the item Gini; neither is 'the' fix, because they measure and trade different things.


## Exercise 10 - Challenge: a reader-side slice -- light vs heavy readers

I split the cohort at the median train-history length and report each group's hit@10 and head-share for three paths. For the two-tower the light readers score higher, but the unpaired standard error of the gap is large: the gap is under two standard errors, so it is a hint, not a finding.

In [11]:
history_length = {r: len(seen_by[r]) for r in cohort}
median_length = float(np.median(list(history_length.values())))
light = [r for r in cohort if history_length[r] <= median_length]
heavy = [r for r in cohort if history_length[r] > median_length]

group_table = pd.DataFrame([
    {
        "path": name,
        "group": group,
        "readers": len(members),
        "hit@10": hit_of({r: lists_by_path[name][r] for r in members}),
        "head_share": head_share([lists_by_path[name][r] for r in members], head),
    }
    for name in ("popularity", "item-item", "two-tower")
    for group, members in (("light", light), ("heavy", heavy))
])


def per_reader_hits(lists, readers):
    # 1.0 / 0.0 per reader: did their list hit an unseen val positive?
    return np.array([1.0 if set(lists[r]) & (val_pos_by[r] - seen_by[r]) else 0.0 for r in readers])


tt_light = per_reader_hits(lists_by_path["two-tower"], light)
tt_heavy = per_reader_hits(lists_by_path["two-tower"], heavy)
tt_gap = float(tt_light.mean() - tt_heavy.mean())
tt_gap_se = math.sqrt(np.std(tt_light, ddof=1) ** 2 / len(tt_light) + np.std(tt_heavy, ddof=1) ** 2 / len(tt_heavy))
gap_is_clear = bool(abs(tt_gap) >= 2 * tt_gap_se)

reader_side_explanation = (
    "In this sample the two-tower's light readers score about 0.07 higher than its heavy readers, but "
    "the gap is under two standard errors, so it is a hint worth watching, not a finding. The slice "
    "itself is the point: an average hit@10 can hide a group of readers who are served worse, and only "
    "a per-group readout with an error bar shows it. Heavy readers give item-item more co-read evidence, "
    "and much of that evidence is about popular books, so their summed similarity leans further toward "
    "the head."
)

print(f"median history length: {median_length} | light readers: {len(light)} | heavy readers: {len(heavy)}")
print(group_table.round(3))
print(f"two-tower light - heavy: {tt_gap:+.3f} (unpaired SE {tt_gap_se:.3f}) | clear at 2 SE: {gap_is_clear}")
print(reader_side_explanation)

group_rows = {(row["path"], row["group"]): row for row in group_table.to_dict("records")}
assert len(light) + len(heavy) == len(cohort)
assert abs(tt_light.mean() - group_rows[("two-tower", "light")]["hit@10"]) < 1e-12, "per-reader hits reproduce hit_of"
assert group_rows[("item-item", "heavy")]["head_share"] > group_rows[("item-item", "light")]["head_share"]
assert not gap_is_clear, "recorded honestly: the two-tower's light-vs-heavy gap is under 2 SE"


median history length: 8.0 | light readers: 273 | heavy readers: 227
         path  group  readers  hit@10  head_share
0  popularity  light      273   0.121       1.000
1  popularity  heavy      227   0.093       1.000
2   item-item  light      273   0.249       0.443
3   item-item  heavy      227   0.256       0.549
4   two-tower  light      273   0.374       0.901
5   two-tower  heavy      227   0.300       0.898
two-tower light - heavy: +0.074 (unpaired SE 0.042) | clear at 2 SE: False
In this sample the two-tower's light readers score about 0.07 higher than its heavy readers, but the gap is under two standard errors, so it is a hint worth watching, not a finding. The slice itself is the point: an average hit@10 can hide a group of readers who are served worse, and only a per-group readout with an error bar shows it. Heavy readers give item-item more co-read evidence, and much of that evidence is about popular books, so their summed similarity leans further toward the head.
